# Comparison of deconvolver calibrators

In [1]:
import os
from pathlib import Path
import json
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from EDA.edautils import plot_deconvolution_results
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator
from EDA.dirichlet_calibration import DirichletCalibrator

%load_ext autoreload
%autoreload 2

## Utility functions

In [2]:
def prod_len_dict_keys(dict_: dict) -> int:
    return reduce(lambda x,y: x*y, list(map(len,dict_.values())), 1)

In [3]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
    ctype_names: list[str] = None,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label, alpha=0.7)

    # plot the cell types names on the x axis, rotated by 90 degrees
    if ctype_names is not None:
        plt.xticks(x_center, ctype_names, rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

In [4]:
def plot_heatmap(
    matrix: np.ndarray,
    title: str = "Heatmap",
    color_bar_label: str = "Probability",
    xlabel="Predicted Class",
    ylabel="True Class",
    x_ticks: list[str] = None,
    y_ticks: list[str] = None,
    vmin: float | None = None,
    vmax: float | None = None,
):
    """Plot a heatmap of the given matrix with cell type names on the axes."""
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(label=color_bar_label)
    plt.xticks(
        ticks=np.arange(matrix.shape[1]),
        labels=x_ticks if x_ticks is not None else [str(i) for i in range(matrix.shape[1])],
        rotation=90,
    )
    plt.yticks(
        ticks=np.arange(matrix.shape[0]),
        labels=y_ticks if y_ticks is not None else [str(i) for i in range(matrix.shape[0])],
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## Data loading

In [5]:
with open("../App/labels_dict.json", "r") as f:
    labels_to_ctype_names = json.load(f)
with open("../App/old_labels_dict.json", "r") as f:
    old_labels_to_ctype_names = json.load(f)
old_labels_to_ctype_names = {int(k): v for k, v in old_labels_to_ctype_names.items()}
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())

In [6]:
# compute the mapping from old labels to new labels (permutation of the labels)
old_to_new_label_mapping = {old_label: ctype_names_to_labels[old_name] for old_label, old_name in old_labels_to_ctype_names.items()}

In [7]:
target_proportions_file = Path("../Data/training_data/mixture_predictions/uxm/target_proportions.npz")
target_prop = np.load(target_proportions_file)["arr_0"]

In [8]:
uxm_pred_folder = Path("../Data/training_data/mixture_predictions/uxm")
uxm_train_file = uxm_pred_folder / "uxm_results_train.npz"
uxm_val_file = uxm_pred_folder / "uxm_results_valid.npz"
uxm_test_file = uxm_pred_folder / "uxm_results_test.npz"

uxm_train_data = np.load(uxm_train_file)["arr_0"]
uxm_val_data = np.load(uxm_val_file)["arr_0"]
uxm_test_data = np.load(uxm_test_file)["arr_0"]

## Compare calibrators

### Linear calibrators

In [ ]:
def wrapper_comp_metrics(test_pred: dict, round_: int=6) -> pd.DataFrame:
    results = {
        method_name: compute_deconvolution_metrics(
            pred=pred,
            target=target_prop,
            class_names=ctype_names_list
        )
        for method_name, pred in test_pred.items()
    }
    results = {
        k: {metric: value for metric, value in v.items() if "per_class" not in metric}
        for k, v in results.items()
    }
    results_df = pd.DataFrame(results).T
    float_cols = ["mae", "mse", "kl", 'max_error', 'cosine_sim', "loa_lower", "loa_upper", "loa_width", "worst_class_loa_lower", "worst_class_loa_upper", "worst_class_loa_width"]
    for col in float_cols:
        results_df[col] = results_df[col].astype(float).round(round_)
    results_df.sort_values("mse", inplace=True)
    return results_df, results

In [ ]:
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, target_prop)
uxm_test_pred_lin_cal_clip_norm, raw_uxm_test_pred_lin_cal  = linear_cal.predict(uxm_test_data, norm_method="clip01-normalize")
uxm_test_pred_lin_cal_clip_norm_no_upper_clip,_  = linear_cal.predict(uxm_test_data, norm_method="clip0-normalize")
uxm_test_pred_lin_cal_softmax,_  = linear_cal.predict(uxm_test_data, norm_method="softmax")
uxm_test_pred_lin_cal_simplex_proj, _ = linear_cal.predict(uxm_test_data, norm_method="simplex-projection")
uxm_test_pred_lin_cal_shift_norm, _ = linear_cal.predict(uxm_test_data, norm_method="shift-normalize")
uxm_test_pred_lin_cal_entmax, _ = linear_cal.predict(uxm_test_data, norm_method="entmax", entmax_alpha=1.5)

In [ ]:
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + Lin cal (clip01-norm)": uxm_test_pred_lin_cal_clip_norm,
        "UXM + Lin cal (clip0-norm)": uxm_test_pred_lin_cal_clip_norm_no_upper_clip,
        "UXM + Lin cal (softmax)": uxm_test_pred_lin_cal_softmax,
        "UXM + Lin cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "UXM + Lin cal (shift-normalize)": uxm_test_pred_lin_cal_shift_norm,
        "UXM + Lin cal (entmax)": uxm_test_pred_lin_cal_entmax
    }
)
results_df

In [ ]:
# we find the index of the sample with the highest MSE for the "UXM + no cal" method
# sample_idx = np.argmax(np.sum((uxm_test_data - target_prop) ** 2, axis=1))
# we find the index of the sample with the highest MAE for the "UXM + no cal" method
sample_idx = np.argmax(np.sum(np.abs(uxm_test_data - target_prop), axis=1))

In [ ]:
# sample_idx = 1
plot_mixtures_pred_vs_true(
    ground_truth_mixture=target_prop[sample_idx],
    predicted_mixtures=[
        uxm_test_data[sample_idx],
        uxm_test_pred_lin_cal_clip_norm[sample_idx],
        uxm_test_pred_lin_cal_clip_norm_no_upper_clip[sample_idx],
        # uxm_test_pred_lin_cal_softmax[sample_idx],
        uxm_test_pred_lin_cal_simplex_proj[sample_idx],
        # uxm_test_pred_lin_cal_shift_norm[sample_idx],
        # uxm_test_pred_lin_cal_entmax[sample_idx]
    ],
    predicted_mixture_labels=[
        "UXM + no cal",
        "UXM + Lin cal (clip01-norm)",
        "UXM + Lin cal (clip0-norm)",
        # "UXM + Lin cal (softmax)",
        "UXM + Lin cal (simplex-projection)",
        # "UXM + Lin cal (shift-normalize)",
        # "UXM + Lin cal (entmax)"
    ],
    title="UXM Predictions with and without Linear Calibration",
    ctype_names=ctype_names_list
)

In [ ]:
np.sort(uxm_test_data[sample_idx])[::-1][:9]

In [ ]:
np.sort(uxm_test_pred_lin_cal_clip_norm_no_upper_clip[sample_idx])[::-1][:9]

In [ ]:
np.sort(uxm_test_pred_lin_cal_simplex_proj[sample_idx])[::-1][:9]

### Learned calibrators

Next step is to try :

- vector scaling on the log prob (what the Dir Cal paper does)
- vector scaling on the prob
- matrix scaling on the log prob (= Dir Cal)
- matrix scaling on the prob (= MS)

with softmax and maybe with sparsemax.

#### Vector scaling on the log prop

In [ ]:
# Vector scaling on the log prob grid search
METHOD = "diagonal"
LOG_TRANSFORM = True
param_grid = {
    "reg_lambda": [0.0, 1e-3, 1e-2, 1e-1],
    "reg_mu": [None],
    "lr": [1e-2],
    "max_iter": [500],
    "scheduler": ["cosine"],
    "batch_size" : [100000],
    "patience": [50],
    "tol": [1e-7]
}

results = []
n_loops = prod_len_dict_keys(param_grid)
best_val_loss = np.inf
best_model_vect_log_dir_cal = None
with tqdm(total=n_loops) as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for reg_mu in param_grid["reg_mu"]:
            for lr in param_grid["lr"]:
                for max_iter in param_grid["max_iter"]:
                    for scheduler in param_grid["scheduler"]:
                        for batch_size in param_grid["batch_size"]:
                            for patience in param_grid["patience"]:
                                for tol in param_grid["tol"]:
                                    vect_log_dir_cal = DirichletCalibrator(
                                        method=METHOD,
                                        lr=lr,
                                        log_transform=LOG_TRANSFORM,
                                        max_iter=max_iter,
                                        batch_size=batch_size,
                                        device="cuda",
                                        scheduler=scheduler,
                                        optimizer="adam",
                                        reg_lambda=reg_lambda,
                                        reg_mu=reg_mu,
                                        verbose=False,
                                        patience=patience,
                                        tol=tol
                                    )
                                    vect_log_dir_cal.fit(
                                        X=uxm_train_data,
                                        y=target_prop,
                                        X_val=uxm_val_data,
                                        y_val=target_prop,
                                        report_every=10
                                    )
                                    results.append({
                                        # Hyperparameters
                                        "reg_lambda": reg_lambda,
                                        "reg_mu": reg_mu,
                                        "lr": lr,
                                        "max_iter": max_iter,
                                        "scheduler": scheduler,
                                        "batch_size": batch_size,
                                        "patience": patience,
                                        "tol": tol,
                                        # Key metrics
                                        "best_train_loss": vect_log_dir_cal.best_metrics_["train_loss"],
                                        "best_val_loss": vect_log_dir_cal.best_metrics_["val_loss"],
                                        "best_train_mse": vect_log_dir_cal.best_metrics_["train_mse"],
                                        "best_val_mse": vect_log_dir_cal.best_metrics_["val_mse"],
                                        "best_epoch": vect_log_dir_cal.best_epoch_,
                                        "n_epochs_trained": vect_log_dir_cal.history_["epoch"][-1]+1,
                                        # Fitted model (not in DataFrame columns, but accessible)
                                    })
                                    if vect_log_dir_cal.best_metrics_["val_loss"] < best_val_loss:
                                        best_val_loss = vect_log_dir_cal.best_metrics_["val_loss"]
                                        best_model_vect_log_dir_cal = vect_log_dir_cal
                                    pbar.update(1)
results_df_vect_log_dir_cal = pd.DataFrame(results)
results_df_vect_log_dir_cal.sort_values("best_val_loss", inplace=True)
results_df_vect_log_dir_cal

In [9]:
vect_log_dir_cal = DirichletCalibrator(
    method="diagonal",
    lr=0.01,
    log_transform=True,
    max_iter=500,
    batch_size=100000,
    device="cuda",
    scheduler="cosine",
    optimizer="adam",
    reg_lambda=0.0,
    reg_mu=None,
    verbose=True,
    patience=50,
    tol=1e-7
)
vect_log_dir_cal.fit(
    X=uxm_train_data,
    y=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
    report_every=10
)

Epoch -1: train_loss = 8.8158402e-02, val_loss = 2.1000276e-01, train_mse = 9.9436630e-05, val_mse = 2.9901757e-04
Epoch 0: train_loss = 8.8158402e-02, val_loss = 2.0844986e-01, train_mse = 9.2213111e-05, val_mse = 2.9445070e-04
Epoch 10: train_loss = 7.4153944e-02, val_loss = 1.9724305e-01, train_mse = 7.0527840e-05, val_mse = 2.7884787e-04
Epoch 20: train_loss = 6.7437338e-02, val_loss = 1.8647964e-01, train_mse = 6.7119644e-05, val_mse = 2.5578651e-04
Epoch 30: train_loss = 6.4707694e-02, val_loss = 1.8221699e-01, train_mse = 7.5580455e-05, val_mse = 2.5893219e-04
Epoch 40: train_loss = 6.3627754e-02, val_loss = 1.8091864e-01, train_mse = 8.2914644e-05, val_mse = 2.6713572e-04
Epoch 50: train_loss = 6.3163654e-02, val_loss = 1.8006707e-01, train_mse = 8.6410394e-05, val_mse = 2.6863022e-04
Epoch 60: train_loss = 6.2924482e-02, val_loss = 1.8009695e-01, train_mse = 8.7681272e-05, val_mse = 2.6939424e-04
Epoch 70: train_loss = 6.2778924e-02, val_loss = 1.8014991e-01, train_mse = 8.773

,method,'diagonal'
,log_transform,True
,reg_lambda,0.0
,reg_mu,None
,optimizer,'adam'
,lr,0.01
,scheduler,'cosine'
,max_iter,500
,batch_size,100000
,patience,50
,tol,1e-07


In [ ]:
vect_scal_log_softmax_cal_test_pred = best_model_vect_log_dir_cal.predict(uxm_test_data)
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + Lin cal (clip-norm)": uxm_test_pred_lin_cal_clip_norm,
        "UXM + Lin cal (clip-norm no upper clip)": uxm_test_pred_lin_cal_clip_norm_no_upper_clip,
        "UXM + Lin cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "UXM + Vect Scal log softmax Cal": vect_scal_log_softmax_cal_test_pred
    }
)
results_df.sort_values("mse")

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
path_to_save = Path("../output/calibrators_comparison") / "vect_scal_log_dir_cal_gridsearch_results_2.csv"
if not os.path.exists(path_to_save):
    results_df.to_csv(path_to_save, index=False)
else:
    print(f"File {path_to_save} already exists. Not saving results to avoid overwriting.")

#### MS on the log prob

In [ ]:
# Matrix scaling on the log prob (Dir cal) grid search
METHOD = "full"
LOG_TRANSFORM = True
param_grid = {
    "reg_lambda": [0.0, 1e-1, 1e-2, 1e-3, 1e-4],
    "reg_mu": [None],
    "lr": [1e-3, 1e-4, 1e-5],
    "max_iter": [500],
    "scheduler": ["cosine"],
    "batch_size" : [100000],
    "patience": [50],
    "tol": [1e-7]
}

results = []
n_loops = prod_len_dict_keys(param_grid)
best_val_loss = np.inf
best_model_mat_log_dir_cal = None
with tqdm(total=n_loops) as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for reg_mu in param_grid["reg_mu"]:
            for lr in param_grid["lr"]:
                for max_iter in param_grid["max_iter"]:
                    for scheduler in param_grid["scheduler"]:
                        for batch_size in param_grid["batch_size"]:
                            for patience in param_grid["patience"]:
                                for tol in param_grid["tol"]:
                                    mat_log_dir_cal = DirichletCalibrator(
                                        method=METHOD,
                                        lr=lr,
                                        log_transform=LOG_TRANSFORM,
                                        max_iter=max_iter,
                                        batch_size=batch_size,
                                        device="cuda",
                                        scheduler=scheduler,
                                        optimizer="adam",
                                        reg_lambda=reg_lambda,
                                        reg_mu=reg_mu,
                                        verbose=False,
                                        patience=patience,
                                        tol=tol
                                    )
                                    mat_log_dir_cal.fit(
                                        X=uxm_train_data,
                                        y=target_prop,
                                        X_val=uxm_val_data,
                                        y_val=target_prop,
                                        report_every=10
                                    )
                                    results.append({
                                        # Hyperparameters
                                        "reg_lambda": reg_lambda,
                                        "reg_mu": reg_mu,
                                        "lr": lr,
                                        "max_iter": max_iter,
                                        "scheduler": scheduler,
                                        "batch_size": batch_size,
                                        "patience": patience,
                                        "tol": tol,
                                        # Key metrics
                                        "best_train_loss": mat_log_dir_cal.best_metrics_["train_loss"],
                                        "best_val_loss": mat_log_dir_cal.best_metrics_["val_loss"],
                                        "best_train_mse": mat_log_dir_cal.best_metrics_["train_mse"],
                                        "best_val_mse": mat_log_dir_cal.best_metrics_["val_mse"],
                                        "best_epoch": mat_log_dir_cal.best_epoch_,
                                        "n_epochs_trained": mat_log_dir_cal.history_["epoch"][-1]+1,
                                        # Fitted model (not in DataFrame columns, but accessible)
                                    })
                                    if mat_log_dir_cal.best_metrics_["val_loss"] < best_val_loss:
                                        best_val_loss = mat_log_dir_cal.best_metrics_["val_loss"]
                                        best_model_mat_log_dir_cal = mat_log_dir_cal
                                    pbar.update(1)
results_df = pd.DataFrame(results)
results_df

In [ ]:
mat_log_dir_cal = DirichletCalibrator(
    method="full",
    lr=1e-8,
    log_transform=True,
    max_iter=500,
    batch_size=200000,
    device="cuda",
    scheduler="cosine",
    optimizer="sgd",
    reg_lambda=1.0,
    reg_mu=0.0,
    init_noise_std=1e-13,
    verbose=True,
    patience=20,
    tol=1e-7
)
mat_log_dir_cal.fit(
    X=uxm_train_data,
    y=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
    report_every=10
)

In [ ]:
np.diag(mat_log_dir_cal.model_.W_delta.detach().cpu().numpy())

In [ ]:
plot_heatmap(
    mat_log_dir_cal.model_.get_W_matrix().detach().cpu().numpy(),
    x_ticks=ctype_names_list,
    y_ticks=ctype_names_list,
    color_bar_label="Weight Value",
    xlabel="unnormalized log prob",
    ylabel="norm prob",
    title="Learned W_delta Matrix (Dirichlet Calibration)",
    vmax=1e-5
)

Matrix scaling does not reach the same perf as vector scaling, we give it up.

#### Vect scaling on the prob with sparsemax

In [ ]:
vect_scal_nolog_sparsemax_cal = DirichletCalibrator(
    method="diagonal",
    lr=0.01,
    log_transform=False,
    max_iter=500,
    batch_size=100000,
    device="cuda",
    scheduler="cosine",
    optimizer="adam",
    reg_lambda=0.0,
    reg_mu=None,
    verbose=True,
    patience=50,
    tol=1e-7,
    normalization="sparsemax"
)
vect_scal_nolog_sparsemax_cal.fit(
    X=uxm_train_data,
    y=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
    report_every=10
)

In [ ]:
# Vector scaling on the log prob grid search
METHOD = "diagonal"
LOG_TRANSFORM = False
NORMALIZATION = "sparsemax"
param_grid = {
    "reg_lambda": [0.0],
    "reg_mu": [None],
    "lr": [0.01, 0.001],
    "max_iter": [500, 1000],
    "scheduler": ["cosine"],
    "batch_size" : [100000],
    "patience": [50],
    "tol": [1e-7]
}

results = []
n_loops = prod_len_dict_keys(param_grid)
best_val_loss = np.inf
best_model_vect_nolog_sparsemax_dir_cal = None
with tqdm(total=n_loops) as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for reg_mu in param_grid["reg_mu"]:
            for lr in param_grid["lr"]:
                for max_iter in param_grid["max_iter"]:
                    for scheduler in param_grid["scheduler"]:
                        for batch_size in param_grid["batch_size"]:
                            for patience in param_grid["patience"]:
                                for tol in param_grid["tol"]:
                                    vect_nolog_sparsemax_dir_cal = DirichletCalibrator(
                                        method=METHOD,
                                        lr=lr,
                                        log_transform=LOG_TRANSFORM,
                                        max_iter=max_iter,
                                        batch_size=batch_size,
                                        device="cuda",
                                        scheduler=scheduler,
                                        optimizer="adam",
                                        reg_lambda=reg_lambda,
                                        reg_mu=reg_mu,
                                        verbose=False,
                                        patience=patience,
                                        tol=tol,
                                        normalization=NORMALIZATION
                                    )
                                    vect_nolog_sparsemax_dir_cal.fit(
                                        X=uxm_train_data,
                                        y=target_prop,
                                        X_val=uxm_val_data,
                                        y_val=target_prop,
                                        report_every=10
                                    )
                                    results.append({
                                        # Hyperparameters
                                        "reg_lambda": reg_lambda,
                                        "reg_mu": reg_mu,
                                        "lr": lr,
                                        "max_iter": max_iter,
                                        "scheduler": scheduler,
                                        "batch_size": batch_size,
                                        "patience": patience,
                                        "tol": tol,
                                        # Key metrics
                                        "best_train_loss": vect_nolog_sparsemax_dir_cal.best_metrics_["train_loss"],
                                        "best_val_loss": vect_nolog_sparsemax_dir_cal.best_metrics_["val_loss"],
                                        "best_train_mse": vect_nolog_sparsemax_dir_cal.best_metrics_["train_mse"],
                                        "best_val_mse": vect_nolog_sparsemax_dir_cal.best_metrics_["val_mse"],
                                        "best_epoch": vect_nolog_sparsemax_dir_cal.best_epoch_,
                                        "n_epochs_trained": vect_nolog_sparsemax_dir_cal.history_["epoch"][-1]+1,
                                        # Fitted model (not in DataFrame columns, but accessible)
                                    })
                                    if vect_nolog_sparsemax_dir_cal.best_metrics_["val_loss"] < best_val_loss:
                                        best_val_loss = vect_nolog_sparsemax_dir_cal.best_metrics_["val_loss"]
                                        best_model_vect_nolog_sparsemax_dir_cal = vect_nolog_sparsemax_dir_cal
                                    pbar.update(1)
results_df = pd.DataFrame(results)
results_df.sort_values("best_val_mse")

In [ ]:
vect_scal_nolog_sparsemax_cal_test_pred = best_model_vect_nolog_sparsemax_dir_cal.predict(uxm_test_data)
results_df, results = wrapper_comp_metrics(
    {
        "UXM + no cal": uxm_test_data,
        "UXM + affine cal (clip01-norm) (OG)": uxm_test_pred_lin_cal_clip_norm,
        "UXM + affine cal (clip0-norm)": uxm_test_pred_lin_cal_clip_norm_no_upper_clip,
        "UXM + affine cal (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "UXM + Vect Scal no log prob + Sparsemax Cal": vect_scal_nolog_sparsemax_cal_test_pred,
        "UXM + Vect Scal log prob + Softmax Cal": vect_scal_log_softmax_cal_test_pred
    }
)
results_df.sort_values("mse")

In [ ]:
results_df.to_csv("../output/calibrators_comparison/global_comparison_1.csv", index=False)